# HW4. Olist: PostgreSQL DML, JOIN, GROUP BY, window functions

Дані: Brazilian E-Commerce (Olist). СУБД: локальний PostgreSQL 16 (pgAdmin), база hw4.

## Завдання 1. Setup і typed-схема Olist

In [3]:
%pip install -q psycopg2-binary sqlalchemy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import zipfile
import getpass
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Пароль не зберігається у файлі: береться зі змінної середовища або вводиться вручну.
DB_URL = os.getenv("HW4_DB_URL")
if not DB_URL:
    DB_URL = f"postgresql+psycopg2://postgres:{getpass.getpass('Пароль користувача postgres: ')}@localhost:5432/hw4"

engine = create_engine(DB_URL, future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


def run(sql_text):
    """Виконує SQL (можна кілька команд). Якщо остання повертає рядки, віддає DataFrame."""
    with engine.begin() as conn:
        res = conn.exec_driver_sql(sql_text)
        if res.returns_rows:
            return pd.DataFrame(res.fetchall(), columns=list(res.keys()))

PostgreSQL 16.15, compiled by Visual C++ build 1944, 64-bit


In [5]:
# CSV Olist взято з архіву Kaggle (archive.zip) і розпаковано локально.
ZIP_PATH = r"C:\Users\PC\Downloads\archive.zip"
EXTRACT_DIR = r"C:\Users\PC\Downloads\olist_data"

dataset_dir = os.getenv("OLIST_DIR")
if not dataset_dir:
    if not os.path.isdir(EXTRACT_DIR):
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(EXTRACT_DIR)
    # шукаємо папку, де лежить olist_customers_dataset.csv (може бути вкладена)
    for root, _, names in os.walk(EXTRACT_DIR):
        if "olist_customers_dataset.csv" in names:
            dataset_dir = root
            break

print(dataset_dir)
print(sorted(f for f in os.listdir(dataset_dir) if f.endswith(".csv")))

C:\Users\PC\Downloads\olist_data
['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [6]:
files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}

raw_counts = {}
for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000)
    raw_counts[table_name] = len(df)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


Raw-таблиці лише проміжний шар (копія CSV). Далі всі завдання працюють з typed-таблицями, у яких є типи та constraints.

In [7]:
run("""
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);
""")

In [8]:
run("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
""")

У вихідному CSV колонки products названо product_name_lenght та product_description_lenght (з помилкою). У typed-схемі використано нормалізовані назви product_name_length і product_description_length, а в SELECT із raw-таблиці фактичні назви з CSV.

In [9]:
run("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;
""")

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


In [10]:
# порівняння з raw-таблицями (кількість рядків має збігатися)
typed = run("""
SELECT 'olist_customers_raw' AS raw_table, COUNT(*) AS typed_rows FROM olist_customers
UNION ALL SELECT 'olist_orders_raw', COUNT(*) FROM olist_orders
UNION ALL SELECT 'olist_order_items_raw', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'olist_products_raw', COUNT(*) FROM olist_products
UNION ALL SELECT 'olist_sellers_raw', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'olist_order_reviews_raw', COUNT(*) FROM olist_order_reviews
""")
typed["raw_rows"] = typed["raw_table"].map(raw_counts)
typed["diff"] = typed["raw_rows"] - typed["typed_rows"]
typed

,raw_table,typed_rows,raw_rows,diff
0,olist_sellers_raw,3095,3095,0
1,olist_products_raw,32951,32951,0
2,olist_customers_raw,99441,99441,0
3,olist_orders_raw,99441,99441,0
4,olist_order_reviews_raw,99224,99224,0
5,olist_order_items_raw,112650,112650,0


Висновок smoke-test: кількість рядків у кожній із шести typed-таблиць збігається з raw (diff = 0), тож під час перенесення нічого не втрачено: усі значення пройшли приведення типів і constraints, зокрема CHECK на order_status і review_score.

In [11]:
# перевірка constraints у typed-таблицях
run("""
SELECT conrelid::regclass AS table_name, conname, contype,
       pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid::regclass::TEXT ~ '^olist_'
  AND conrelid::regclass::TEXT !~ '_raw$'
ORDER BY table_name, contype;
""")

,table_name,conname,contype,definition
0,olist_customers,olist_customers_pkey,p,PRIMARY KEY (customer_id)
1,olist_orders,olist_orders_order_status_check,c,CHECK ((order_status = ANY (ARRAY['created'::t...
2,olist_orders,olist_orders_customer_id_fkey,f,FOREIGN KEY (customer_id) REFERENCES olist_cus...
3,olist_orders,olist_orders_pkey,p,PRIMARY KEY (order_id)
4,olist_products,olist_products_product_weight_g_check,c,CHECK (((product_weight_g IS NULL) OR (product...
5,olist_products,olist_products_product_length_cm_check,c,CHECK (((product_length_cm IS NULL) OR (produc...
6,olist_products,olist_products_product_height_cm_check,c,CHECK (((product_height_cm IS NULL) OR (produc...
7,olist_products,olist_products_product_width_cm_check,c,CHECK (((product_width_cm IS NULL) OR (product...
8,olist_products,olist_products_product_name_length_check,c,CHECK (((product_name_length IS NULL) OR (prod...
9,olist_products,olist_products_product_description_length_check,c,CHECK (((product_description_length IS NULL) O...


## Завдання 2. DML-запити з PostgreSQL-фічами

### 2.1. INSERT ... RETURNING: alert для продавця з найбільшою кількістю запізнень

In [12]:
run("""
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, details, created_at;
""")

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the lo...,2026-10-11 18:18:34.395332+03:00


Що змінює запит: створює таблицю seller_alerts і додає в неї один alert для продавця з найбільшою кількістю запізнених замовлень. Це продавець 4a3ca9315b744ce9f8e9374361493884: у нього 172 запізнені замовлення з 1 806 (близько 9,5%).

RETURNING одразу повертає створений рядок разом зі згенерованими alert_id і created_at, тому результат можна використати як audit-log у notebook без окремого SELECT.

Рахуємо пари seller_id × order_id через SELECT DISTINCT, щоб замовлення з кількома позиціями одного продавця не рахувалося кілька разів. Для повторного запуску таблиця спершу видаляється через DROP TABLE IF EXISTS, тому дублів alert-ів не виникає.

### 2.2. INSERT ... ON CONFLICT DO UPDATE: idempotent upsert seller_score

In [13]:
run("""
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;
""")

,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-10-11 18:18:34.653578+03:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-10-11 18:18:34.653578+03:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-10-11 18:18:34.653578+03:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-10-11 18:18:34.653578+03:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-10-11 18:18:34.653578+03:00
...,...,...,...,...,...
3090,b2eecf5ea250510da76590ca79d60e5d,5.00,9,gold,2026-10-11 18:18:34.653578+03:00
3091,eae9af4811c294c56795d70e715b7337,5.00,1,gold,2026-10-11 18:18:34.653578+03:00
3092,1c83418358f249c9d6affa52d579db0e,5.00,1,gold,2026-10-11 18:18:34.653578+03:00
3093,845e8036c3f7bf62099352f5ed910240,3.95,21,silver,2026-10-11 18:18:34.653578+03:00


In [14]:
# підсумок за рівнями та перевірка ідемпотентності: кількість рядків після upsert
run("""
SELECT tier, COUNT(*) AS n_sellers, ROUND(AVG(avg_rating), 2) AS mean_rating
FROM seller_score
GROUP BY tier
ORDER BY tier NULLS LAST;
""")

,tier,n_sellers,mean_rating
0,bronze,539,2.30
1,gold,983,4.83
2,silver,1568,4.06
3,NaN,5,None


Чому seller_orders використовує SELECT DISTINCT: в order_items один продавець може мати кілька позицій в одному замовленні. Без DISTINCT таке замовлення рахувалося б кілька разів.

Чому review спочатку агрегується до рівня order_id: в одного замовлення може бути кілька відгуків (у даних таких замовлень 547), а після JOIN із seller_orders кожне замовлення має дати одне значення оцінки. Якщо ж приєднати відгуки напряму до order_items, виходить 112 372 рядки замість 99 247 пар seller × order із відгуком, і середня оцінка змінюється у 1 264 із 3 090 продавців (максимальна різниця 1,67 бала).

Чому ON CONFLICT (seller_id) DO UPDATE робить запуск повторюваним: якщо рядок для продавця вже є, він оновлюється (avg_rating, n_reviews, tier, updated_at), а не дублюється, і таблиця завжди містить 3 095 рядків.

Чому sellers без review мають avg_rating = NULL, n_reviews = 0, tier = NULL: LEFT JOIN залишає продавця без збігів, COUNT по порожніх значеннях дає 0, а AVG по порожній множині дає NULL. Таких продавців 5.

Розподіл за рівнями: gold 983, silver 1 568, bronze 539, без рівня 5 (разом 3 095).

Обмеження датасету: review в Olist прив'язаний до замовлення, а не до окремого продавця. Якщо в замовленні товари кількох продавців, один і той самий order-level review потрапляє до кожного з них. Це прийнятне навчальне спрощення, але його треба враховувати в інтерпретації.

### 2.3. UPDATE ... FROM: прапорець запізнення в olist_orders

In [15]:
run("""
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;

WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;
""")

,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False
...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,False
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,False
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,False
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,False


In [16]:
run("""
SELECT is_late, COUNT(*) AS n_orders
FROM olist_orders
GROUP BY is_late
ORDER BY is_late;
""")

,is_late,n_orders
0,False,92906
1,True,6535


Що змінює запит: додає колонку is_late і виставляє її для всіх 99 441 замовлень. Прострочено 6 535 замовлень (близько 6,6%), решта 92 906 доставлені вчасно або ще не мають фактичної дати доставки.

Запит ідемпотентний: ADD COLUMN IF NOT EXISTS не падає вдруге, а UPDATE щоразу перераховує is_late з дат, а не дописує TRUE для частини рядків. RETURNING показує, які саме рядки оновлено.

Зверніть увагу: для замовлень без дати доставки (скасовані, у дорозі) прапорець дорівнює FALSE, бо запізнення довести не можна. Це спрощення, яке треба пам'ятати під час аналізу.

### 2.4. DELETE ... RETURNING: контрольоване видалення застарілих alert-ів

In [17]:
run("""
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;
""")

,alert_id,seller_id,alert_type,created_at


Результат порожній, і це допустимо: усі alert-и створені щойно (created_at = NOW()), а умова видаляє лише записи старші за 30 днів. Запит безпечний для повторного запуску: він видаляє лише застарілі alert-и і не чіпає свіжі.

DELETE ... RETURNING у production корисний тим, що одразу показує, які саме рядки видалено, і ці дані можна записати в архів або лог.

## Завдання 3. CREATE TABLE customer_segments

Спочатку дедуплікований customer dimension: один рядок на реального покупця (customer_unique_id).

In [18]:
run("""
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;
""")

In [19]:
run("""
SELECT
    (SELECT COUNT(*) FROM olist_customers)    AS customer_id_rows,
    (SELECT COUNT(*) FROM olist_customer_dim) AS unique_customers;
""")

,customer_id_rows,unique_customers
0,99441,96096


In [20]:
run("""
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
""")

In [21]:
run("""
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;
""")

,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
...,...,...,...,...,...,...
96091,fffcf5a5ff07b0908bd4e2dbc735a684,VIP,5,497,2067.42,1
96092,fffea47cd6d3cc0a88bd621562a9d061,inactive,2,312,84.58,1
96093,ffff371b4d645b6ecea244b27531430a,regular,2,618,112.46,1
96094,ffff5962728ec6157033ef9805bacc48,regular,2,169,133.69,1


In [22]:
run("""
SELECT segment_name, COUNT(*) AS n_customers,
       ROUND(AVG(monetary_value), 2) AS avg_monetary,
       ROUND(AVG(n_orders), 2) AS avg_orders
FROM customer_segments
GROUP BY segment_name
ORDER BY n_customers DESC;
""")

,segment_name,n_customers,avg_monetary,avg_orders
0,regular,45982,224.57,1.07
1,inactive,40032,59.28,1.00
2,new,8891,140.39,1.00
3,VIP,1191,1591.81,1.11


In [23]:
run("""
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
""")

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...


Чому customer_segments використовує customer_unique_id: у Olist customer_id створюється окремо для кожного замовлення, тому один покупець має кілька customer_id. У даних 99 441 customer_id, але лише 96 096 реальних покупців, і 2 997 із них мають більше одного customer_id. Якщо рахувати RFM за customer_id, frequency майже завжди дорівнює 1.

Чому olist_customer_dim потрібна для FK: customer_unique_id у olist_customers повторюється, тому на нього не можна поставити FOREIGN KEY. У dim-таблиці він є PRIMARY KEY, тож customer_segments посилається на унікальний ключ.

Чому reference_date краще брати з датасету: Olist є історичним (2016 до 2018), а reference_date = MAX(order_purchase_timestamp) + 1 = 2018-10-18. Якби використати CURRENT_DATE, recency_days перевищував би 2 900 днів і змінювався б щодня, тож сегментація була б невідтворюваною.

Як ON CONFLICT робить сегментацію повторюваною: для існуючого customer_unique_id рядок оновлюється, а не дублюється (UNIQUE (customer_unique_id)), і assigned_at фіксує час останнього перерахунку.

Результат: regular 45 982 (47,9%), inactive 40 032 (41,7%), new 8 891 (9,3%), VIP 1 191 (1,2%). Лише 2 997 покупців мають більше одного замовлення, тому сегмент здебільшого визначається сумою витрат.

## Завдання 4. JOIN-запити з business-interpretation

### 4.1. INNER JOIN на чотирьох таблицях: топ категорій за revenue і seller-state

Business-question: які категорії товарів і штати продавців дають найбільший revenue у delivered-замовленнях?

In [24]:
run("""
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;
""")

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


Інтерпретація: усі десять рядків топу належать продавцям зі штату SP. Лідирують cama_mesa_banho (8 214 замовлень, 1,07 млн revenue) і relogios_presentes (1,02 млн). Продавці з SP дають близько 9,96 млн із 15,42 млн revenue у delivered-замовленнях, тобто приблизно 65%, тож маркетплейс сильно залежить від одного штату. INNER JOIN залишає лише позиції, у яких є і замовлення, і товар, і продавець.

### 4.2. LEFT JOIN + COALESCE: customer-level features без множення review

Business-question: які customer-level features можна побудувати для реального покупця на рівні customer_unique_id?

In [25]:
run("""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
    COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.00
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.00
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.00
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.00
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.00
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.00
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.00
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.00
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.00
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.00


Інтерпретація: у топі за витратами переважають покупці з одним дорогим замовленням (наприклад, 13 664,08 для одного замовлення в RJ), тобто високі витрати зазвичай не означають повторні покупки. review_per_order захищає від множення review score через позиції замовлення: кожне замовлення дає одну оцінку.

Застереження щодо COALESCE: покупці без відгуку отримують avg_review_score = 0,00, але шкала відгуків починається з 1. Для ML краще залишити NULL або додати прапорець has_review, інакше модель сприйме відсутність відгуку як найгіршу оцінку.

### 4.3. FULL OUTER JOIN: які штати присутні серед клієнтів і продавців

Business-question: які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

In [26]:
run("""
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customers,
    s.n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state;
""")

,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


Інтерпретація: у клієнтів представлено 27 штатів, у продавців 23. У обох групах є 23 штати, лише серед клієнтів 4 штати (AL, AP, RR, TO), штатів лише серед продавців немає. Покупці з цих чотирьох штатів отримують товари від продавців з інших штатів, а отже, ймовірно, довше чекають на доставку. FULL OUTER JOIN потрібен, щоб побачити обидві сторони: INNER JOIN втратив би ці чотири штати.

### 4.4. SELF JOIN: пари продавців з одного штату, але з різних міст

Business-question: які seller-пари можуть бути кандидатами для регіонального порівняння в межах одного штату?

In [27]:
run("""
SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20;
""")

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


In [28]:
# скільки всього таких пар (LIMIT 20 показує лише початок)
run("""
SELECT COUNT(*) AS n_pairs
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city;
""")

,n_pairs
0,1572729


Інтерпретація: умова s1.seller_id < s2.seller_id прибирає дзеркальні пари (A, B) і (B, A) та пари продавця із самим собою, а IS DISTINCT FROM коректно порівнює міста навіть за NULL. Загалом знайдено 1 572 729 пар, бо лише у штаті SP 1 849 продавців. Для реального регіонального порівняння такий набір потрібно додатково звужувати, наприклад за категорією товарів або обсягом продажів.

### 4.5. Anti-join: замовлення без відгуку

Business-question: які замовлення не мають жодного review-запису?

In [29]:
run("""
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 20;
""")

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


In [30]:
run("""
SELECT o.order_status, COUNT(*) AS orders_without_review
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
GROUP BY o.order_status
ORDER BY orders_without_review DESC;
""")

,order_status,orders_without_review
0,delivered,646
1,shipped,75
2,canceled,20
3,unavailable,14
4,processing,6
5,invoiced,5
6,created,2


Інтерпретація: 768 замовлень не мають жодного review: 646 delivered, 75 shipped, 20 canceled, 14 unavailable, 6 processing, 5 invoiced, 2 created. Тобто більшість з них доставлені, і клієнт просто не залишив відгук, а решта не дійшли до клієнта. Для rating-метрик важливо, що відсутність відгуку не дорівнює поганій оцінці.

## Завдання 5. GROUP BY, HAVING і агрегати

### 5.1. Revenue per product category з фільтром по revenue

In [31]:
run("""
SELECT
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
    SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
    AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST;
""")

,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8836,1258681.34,182566.73,130.16
1,relogios_presentes,5624,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30
3,esporte_lazer,7720,988048.97,168607.51,114.34
4,informatica_acessorios,6689,911954.32,147318.08,116.51
5,moveis_decoracao,6449,729762.49,172749.30,87.56
6,cool_stuff,3632,635290.85,84039.10,167.36
7,utilidades_domesticas,5884,632248.66,146149.11,90.79
8,automotivo,3897,592720.11,92664.21,139.96
9,ferramentas_jardim,3518,485256.46,98962.75,111.63


Інтерпретація: фільтр HAVING SUM(price) > 10000 залишає 53 категорії. За revenue лідирує beleza_saude (1,26 млн), другою йде relogios_presentes (1,21 млн) із найвищою середньою ціною серед лідерів (201,14), а за кількістю замовлень першою є cama_mesa_banho (9 417) із нижчою середньою ціною (93,30). Тобто лідери за revenue і за обсягом замовлень різні.

WHERE і HAVING: WHERE фільтрує рядки до агрегації, а HAVING фільтрує групи після неї. Умова SUM(price) > 10000 можлива лише в HAVING, бо сума відома тільки після GROUP BY.

### 5.2. AOV per customer-state для штатів із достатньою кількістю замовлень

In [32]:
run("""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_state
)
SELECT
    customer_state,
    COUNT(*) AS n_orders,
    SUM(order_total)::NUMERIC(12, 2) AS gmv,
    (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST;
""")

,customer_state,n_orders,gmv,aov
0,PB,532,140987.81,265.01
1,AL,411,96229.40,234.13
2,RO,247,57558.02,233.03
3,PA,970,217647.11,224.38
4,TO,279,61354.42,219.91
5,PI,493,108132.28,219.34
6,SE,345,73032.32,211.69
7,RN,482,101895.08,211.40
8,CE,1327,275606.30,207.69
9,MT,903,186168.96,206.17


Інтерпретація: HAVING COUNT(*) >= 100 залишає 24 зі 27 штатів, у яких достатньо замовлень для стабільного AOV. Найвищий AOV у PB (265,01, 532 замовлення), а найнижчий у SP (143,12, 41 375 замовлень). Віддалені штати мають вищий середній чек, а штат із найбільшим обсягом дає найнижчий, тобто AOV і обсяг замовлень рухаються в протилежних напрямках.

### 5.3. Категорії, які купував реальний покупець, через STRING_AGG

In [33]:
run("""
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 1
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

,customer_unique_id,n_orders,total_spend,categories
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.00,telefonia_fixa
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00,telefonia_fixa
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.00,utilidades_domesticas
4,459bef486812aa25204be022145caa62,1,6729.00,pcs
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00,artes
6,4007669dec559734d6f53e029e360987,1,5934.60,agro_industria_e_comercio
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00,eletroportateis
8,5d0a2980b292d049061542014e8960bf,1,4599.90,cool_stuff
9,48e1ac109decbb87765a3eade6854098,1,4590.00,pcs


Інтерпретація: STRING_AGG(DISTINCT ...) збирає унікальні категорії покупця в один рядок, а ORDER BY усередині агрегату робить результат детермінованим. COALESCE замінює відсутню категорію на (unknown). Найбільші витрати (13 440,00 за ціною товарів без доставки) належать покупцю, який купив у категорії telefonia_fixa. HAVING COUNT(DISTINCT o.order_id) >= 1 фільтрує групи після агрегації, тут воно нікого не відсікає, бо JOIN уже залишив лише покупців із замовленнями.

## Завдання 6. Set operations, window function і Reflection

### 6.1. UNION: усі state-коди, що є у клієнтів або продавців

In [34]:
run("""
SELECT customer_state AS state, 'customer' AS role
FROM olist_customers

UNION

SELECT seller_state AS state, 'seller' AS role
FROM olist_sellers
ORDER BY state, role;
""")

,state,role
0,AC,customer
1,AC,seller
2,AL,customer
3,AM,customer
4,AM,seller
5,AP,customer
6,BA,customer
7,BA,seller
8,CE,customer
9,CE,seller


### 6.2. INTERSECT: штати, у яких є і клієнти, і продавці

In [35]:
run("""
SELECT customer_state AS state
FROM olist_customers

INTERSECT

SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;
""")

,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


Чому UNION і INTERSECT: у 6.1 у вибірці є колонка role, тому рядки ('AL', 'customer') і ('AL', 'seller') різні, і UNION не зводить їх разом. Вийшло 50 рядків: 27 штатів клієнтів плюс 23 штати продавців. UNION обрано замість UNION ALL, щоб гарантовано прибрати повтори всередині кожної групи (тут вони вже унікальні, тож результат збігається з UNION ALL). INTERSECT у 6.2 повертає 23 штати, де є і клієнти, і продавці, що збігається зі статусом both у запиті 4.3 (27 мінус 4 штати лише з клієнтами).

### 6.3. ROW_NUMBER(): найновіше замовлення кожного реального покупця

In [36]:
run("""
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;
""")

,customer_unique_id,customer_id,order_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12


Пояснення: PARTITION BY customer_unique_id ділить замовлення за реальним покупцем, ORDER BY ... DESC нумерує їх від найновішого, тому rn = 1 дає останнє замовлення кожного покупця (без LIMIT це 96 096 рядків). ROW_NUMBER обрано замість RANK, бо потрібен рівно один рядок навіть за однакових timestamp, а додатковий order_id DESC робить вибір детермінованим.

## Reflection

1. Найскладнішим був SELF JOIN. Умова s1.seller_id < s2.seller_id прибирає дзеркальні пари, але результат усе одно вийшов величезним (близько 1,57 млн пар через 1 849 продавців зі штату SP), тож без додаткових фільтрів він непридатний для аналізу.

2. Найважче було сформулювати business-question для customer-level features (запит 4.2): спершу треба визначити, що таке покупець і на якому рівні рахувати ознаки.

3. Для production-ETL найкорисніший ON CONFLICT DO UPDATE: він робить наповнення ідемпотентним, тож пайплайн можна безпечно перезапускати після збою без дублікатів. Constraints теж важливі, але вони лише захищають таблицю, а повторюваність запуску дає саме upsert.

4. Window functions у ML-проєкті я б використала для останнього замовлення клієнта (ROW_NUMBER), інтервалу між покупками (LAG), ковзних сум за 30 днів і рангу продавця в категорії. Усе це ознаки, які модель не може отримати зі звичайного GROUP BY, бо вони залежать від порядку подій.

5. Data-quality: 768 замовлень без відгуку, 610 товарів без категорії й розмірів, 814 повторів review_id, 547 замовлень із кількома відгуками, 8 delivered-замовлень без дати доставки, 58 247 відгуків без тексту коментаря.

6. customer_id в Olist створюється для кожного замовлення, тому один покупець має кілька customer_id і frequency майже завжди дорівнює 1; customer_unique_id ідентифікує людину.

7. JOIN з order_items розмножує один відгук на кількість позицій; на рівні seller_id × order_id відгук рахується один раз, тож оцінка не спотворюється (у 1 264 продавців вона відрізнялася). Тому грануляцію треба фіксувати до JOIN, а не після нього.

## Підсумки

У роботі створено typed-схему з ключами й обмеженнями, виконано DML-операції PostgreSQL (RETURNING, ON CONFLICT, UPDATE ... FROM, DELETE ... RETURNING), побудовано customer-level сегментацію на рівні customer_unique_id, проконтрольовано грануляцію під час JOIN, пораховано бізнес-метрики через GROUP BY і HAVING, застосовано set operations та window function.